In [1]:
!curl -O https://raw.githubusercontent.com/esensato/ssa-2026-01/refs/heads/main/carros-fipe.csv

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   5082 100   5082   0      0  40885      0                              0
100   5082 100   5082   0      0  40780      0                              0
100   5082 100   5082   0      0  40734      0                              0


In [2]:
import pandas as pd

import numpy as np

import re

from datetime import datetime

import requests

df = pd.read_csv('carros-fipe.csv', sep=';')

df_orig = df.copy()

df = df.drop(columns=['registro_id'])


def padronizar_ano_modelo(ano):

    match = re.match(r'(\d{4})[- ]?(\d{1,2})', str(ano).strip())

    if match:
        return f"{match.group(1)}-{int(match.group(2)):02d}"

    return np.nan


df['ano_modelo'] = df['ano_modelo'].apply(padronizar_ano_modelo)


def padronizar_valor(valor):

    valor = str(valor).replace('R$', '').replace(' ', '').replace('.', '').replace(',', '.')

    valor = re.sub(r'[^0-9.]', '', valor)

    partes = valor.split('.')

    if len(partes) > 2:
        valor = ''.join(partes[:-1]) + '.' + partes[-1]

    try:
        return float(valor)
    except:
        return np.nan


df['valor_venda'] = df['valor_venda'].apply(padronizar_valor)

df['taxa_servico'] = df['taxa_servico'].apply(padronizar_valor)

df['valor_total'] = df['valor_total'].apply(padronizar_valor)


def padronizar_data_venda(data):

    for fmt in ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d', '%d.%m.%Y']:
        try:
            return datetime.strptime(str(data).strip(), fmt).strftime('%d/%m/%Y')
        except:
            continue

    return np.nan


df['data_venda'] = df['data_venda'].apply(padronizar_data_venda)


def padronizar_km(km):

    km = str(km).replace('km', '').replace('.', '').replace('-', '').replace('/', '').replace(' ', '')

    km = re.sub(r'[^0-9]', '', km)

    try:
        return int(km)
    except:
        return np.nan


df['km_rodados'] = df['km_rodados'].apply(padronizar_km)


def padronizar_placa(placa):

    placa = str(placa).upper().replace('-', '').replace(' ', '')

    placa = re.sub(r'[^A-Z0-9]', '', placa)

    return placa if placa else np.nan


df['placa'] = df['placa'].apply(padronizar_placa)


def padronizar_cpf(cpf):

    cpf = re.sub(r'[^0-9]', '', str(cpf))

    return cpf if len(cpf) == 11 else np.nan


df['cpf'] = df['cpf'].apply(padronizar_cpf)

df['cliente'] = df['cliente'].str.title()

df['estado'] = df['estado'].str.upper().str.replace(' ', '')

obrigatorios = ['placa', 'cpf']

descartados = df[df[obrigatorios].isnull().any(axis=1)]

if not descartados.empty:

    print('Registros descartados por campos obrigatórios:')

    print(descartados)


df = df.drop(descartados.index)

descartados.to_csv('carros-fipe-descarte.csv', sep=';', index=False)

Registros descartados por campos obrigatórios:
   marca_id modelo_id ano_modelo  data_venda  valor_venda  taxa_servico  \
24       21      4828    2010-05  10/07/2025      45990.0         900.0   

    valor_total  km_rodados    placa       cliente  cpf estado  
24      46890.0       98000  STW8V90  Marcos Alves  NaN     SP  


In [3]:
if not df.empty:

    df.iloc[0, df.columns.get_loc('cliente')] = 'Gustavo Telles'

In [4]:
def consultar_fipe(marca_id, modelo_id, ano_modelo):

    url = f"https://parallelum.com.br/fipe/api/v1/carros/marcas/{marca_id}/modelos/{modelo_id}/anos/{ano_modelo}"

    try:

        resp = requests.get(url, timeout=5)

        if resp.status_code == 200:

            dados = resp.json()

            valor = str(dados.get('Valor', ''))

            valor = re.sub(r'[^0-9,]', '', valor)

            valor = valor.replace('.', '').replace(',', '.')

            partes = valor.split('.')

            if len(partes) > 2:

                valor = ''.join(partes[:-1]) + '.' + partes[-1]

            try:

                preco = float(valor)

            except:

                preco = np.nan

            marca = dados.get('Marca', '')

            return preco, marca

        else:

            return np.nan, ''

    except Exception as e:

        print(f'Erro na consulta FIPE para marca_id={marca_id}, modelo_id={modelo_id}, ano_modelo={ano_modelo}: {e}')

        return np.nan, ''


precos = []

marcas = []

descartados_endpoint = []

for idx, row in df.iterrows():

    preco_fipe, marca = consultar_fipe(row['marca_id'], row['modelo_id'], row['ano_modelo'])

    if np.isnan(preco_fipe):

        print(f'Descarte endpoint: linha {idx}, marca_id={row["marca_id"]}, modelo_id={row["modelo_id"]}, ano_modelo={row["ano_modelo"]}')

        descartados_endpoint.append(row)

        continue

    precos.append(preco_fipe)

    marcas.append(marca)


df = df.iloc[:len(precos)]

df['preco_fipe'] = precos

df['marca'] = marcas

df['lucro'] = df['preco_fipe'] - df['valor_venda']


if descartados_endpoint:

    pd.DataFrame(descartados_endpoint).to_csv('carros-fipe-descarte-endpoint.csv', sep=';', index=False)

Descarte endpoint: linha 5, marca_id=21, modelo_id=4828, ano_modelo=2010-06
Descarte endpoint: linha 6, marca_id=23, modelo_id=1043, ano_modelo=2020-01
Descarte endpoint: linha 8, marca_id=23, modelo_id=6831, ano_modelo=2020-01
Descarte endpoint: linha 9, marca_id=22, modelo_id=9120, ano_modelo=2019-03
Descarte endpoint: linha 10, marca_id=22, modelo_id=9120, ano_modelo=2019-03
Descarte endpoint: linha 11, marca_id=22, modelo_id=9120, ano_modelo=2019-03
Descarte endpoint: linha 12, marca_id=44, modelo_id=7710, ano_modelo=2017-01
Descarte endpoint: linha 13, marca_id=44, modelo_id=XXXX, ano_modelo=2017-01
Descarte endpoint: linha 14, marca_id=44, modelo_id=7710, ano_modelo=2017-01
Descarte endpoint: linha 15, marca_id=21, modelo_id=5940, ano_modelo=2014-01
Descarte endpoint: linha 16, marca_id=59, modelo_id=7710, ano_modelo=2017-01
Descarte endpoint: linha 17, marca_id=44, modelo_id=9120, ano_modelo=2019-03
Descarte endpoint: linha 18, marca_id=23, modelo_id=4828, ano_modelo=2010-05
Des

In [5]:
arquivo_saida = 'carros-fipe-saida.csv'

df.to_csv(arquivo_saida, sep=';', index=False)

In [6]:
df['ano'] = df['ano_modelo'].str.split('-').str[0]

anos_ordem = sorted(df['ano'].unique())

pivot = df.pivot_table(index='marca', columns='ano', values='valor_venda', aggfunc='sum', fill_value=0)

pivot = pivot[anos_ordem]

pivot.to_csv('total_por_marca.csv', sep=';')

In [7]:
arquivos_gerados = 'Arquivos gerados: carros-fipe-saida.csv, carros-fipe-descarte.csv, carros-fipe-descarte-endpoint.csv, total_por_marca.csv'

print(arquivos_gerados)

Arquivos gerados: carros-fipe-saida.csv, carros-fipe-descarte.csv, carros-fipe-descarte-endpoint.csv, total_por_marca.csv
